# UrbanMind AI — SMOTE Model Benchmark (Colab)

**الهدف:** تدريب ومقارنة موديلات قابلة للإضافة على Targets الخمسة، مع استخدام `SMOTE` الحقيقي بدل الدالة اليدوية `balance_training_data_smote_like`.

## ترتيب التشغيل
1. تثبيت `imbalanced-learn` وتحميل ملف CSV.
2. بناء الخصائص والـ Targets الخمسة مع الاحتفاظ بتقسيمة `split` الأصلية لو موجودة.
3. تعريف كل الموديلات وإعداداتها في `MODEL_CONFIGS`.
4. لكل Target: اختيار الإعداد الأفضل على Validation، ثم إعادة التدريب على كامل Training، ثم تقييم واحد نهائي على Test.
5. حفظ جدول النتائج والتوقعات وأفضل الإعدادات كملفات CSV/JSON.

> **مهم:** جدول النتائج التاريخية في الخلية التالية من الـ Notebook القديم، وليس نتائج التشغيل الجديد. بعد تشغيل هذا الملف، اعتمد أرقام `smote_model_comparison.csv` للمقارنة الجديدة. تجهيز البيانات وSMOTE داخل `imblearn.Pipeline` حتى لا يحدث resampling للـ Validation أو Test.


In [ ]:
# مرجع: أفضل نتائج مسجلة في الـ Notebook السابق (للمقارنة فقط، ليست نتائج SMOTE الجديد)
previous_best_results = [
    {"Target": "1_Traffic_Susceptibility_Class", "Previous_Best_Model": "Logistic Regression (Tuned + Poly)", "Accuracy": 0.9126, "Macro_F1": 0.9135, "ROC_AUC": 0.9837},
    {"Target": "2_Transit_Availability_Binary", "Previous_Best_Model": "Logistic Regression (Tuned + Poly)", "Accuracy": 0.9393, "Macro_F1": 0.8908, "ROC_AUC": 0.9753},
    {"Target": "3_High_Transit_Accessibility", "Previous_Best_Model": "Logistic Regression (Tuned + Poly)", "Accuracy": 0.9842, "Macro_F1": 0.9781, "ROC_AUC": 0.9981},
    {"Target": "4_Public_Green_Space_Binary", "Previous_Best_Model": "Logistic Regression (Tuned + Poly)", "Accuracy": 0.8935, "Macro_F1": 0.7590, "ROC_AUC": 0.9310},
    {"Target": "5_High_NO2_Air_Pollution", "Previous_Best_Model": "MLP Classifier (baseline features)", "Accuracy": 0.9418, "Macro_F1": 0.9417, "ROC_AUC": 0.9779},
]
import pandas as pd
pd.DataFrame(previous_best_results)


,Target,Previous_Best_Model,Accuracy,Macro_F1,ROC_AUC
0,1_Traffic_Susceptibility_Class,Logistic Regression (Tuned + Poly),0.9126,0.9135,0.9837
1,2_Transit_Availability_Binary,Logistic Regression (Tuned + Poly),0.9393,0.8908,0.9753
2,3_High_Transit_Accessibility,Logistic Regression (Tuned + Poly),0.9842,0.9781,0.9981
3,4_Public_Green_Space_Binary,Logistic Regression (Tuned + Poly),0.8935,0.7590,0.9310
4,5_High_NO2_Air_Pollution,MLP Classifier (baseline features),0.9418,0.9417,0.9779


## 1. تثبيت المكتبات والإعدادات

شغّل الخلايا بالترتيب. في Colab، لو ملف CSV مش موجود في المسار المحدد، هتظهر نافذة لرفع الملف.


In [ ]:
%pip -q install imbalanced-learn


In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier  # مثال جاهز لو حبيت تضيفه كموديل ثالث
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")

# ================================
# CONFIG — عدّل الإعدادات من هنا
# ================================
DATA_PATH = "/content/urbanmind_final_model_ready.csv"
RANDOM_STATE = 42
VALIDATION_SIZE = 0.20       # جزء من الـ Training لاختيار أفضل إعداد
TEST_SIZE_IF_NO_SPLIT = 0.20 # يستخدم فقط لو عمود split غير موجود/غير صالح
SMOTE_K_NEIGHBORS = 5
USE_POLYNOMIAL_FEATURES = True
POLY_DEGREE = 2
SELECT_K_BEST = 120

# تحميل الملف من جهازك في Colab لو مش موجود في DATA_PATH
if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print("ارفع ملف urbanmind_final_model_ready.csv أو ملف CSV بنفس البنية.")
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_candidates:
            raise FileNotFoundError("لم يتم رفع أي ملف CSV.")
        preferred = [name for name in csv_candidates if Path(name).name == "urbanmind_final_model_ready.csv"]
        DATA_PATH = preferred[0] if preferred else csv_candidates[0]
    except ImportError as exc:
        raise FileNotFoundError(
            f"لم أجد الملف في {DATA_PATH}. عدّل DATA_PATH للمسار الصحيح أو شغّل الملف على Google Colab وارفع CSV."
        ) from exc

df = pd.read_csv(DATA_PATH).reset_index(drop=True)
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]:,} columns")


ارفع ملف urbanmind_final_model_ready.csv أو ملف CSV بنفس البنية.


Saving urbanmind_final_model_ready.csv to urbanmind_final_model_ready.csv
Dataset loaded: 3,489 rows × 120 columns


## 2. هندسة الخصائص وبناء الـ Targets

- نستخدم عمود `split` إذا كان يحتوي على صفوف `train` وصفوف أخرى للاختبار.
- لو التقسيمة الأصلية غير موجودة، ننشئ Holdout ثابتًا، مع محاولة Stratification بناءً على Target المرور.
- عتبات الـ Median الخاصة بسهولة الوصول وتلوث NO₂ تُحسب من Training فقط.
- القيم المفقودة في الـ Target تُستبعد من تقييم ذلك الـ Target بدل تحويلها ضمنيًا إلى الفئة 0.


In [ ]:
# ================================
# تحديد تقسيمة Training / Test الأصلية
# ================================
def get_numeric_column(frame, column):
    if column not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")

split_values = df["split"].astype(str).str.strip().str.lower() if "split" in df.columns else pd.Series("", index=df.index)
source_train_mask = split_values.eq("train")
source_test_mask = ~source_train_mask
has_usable_split = source_train_mask.any() and source_test_mask.any()

if has_usable_split:
    train_mask = source_train_mask.copy()
    test_mask = source_test_mask.copy()
    print(f"Using existing split column: train={int(train_mask.sum())}, test={int(test_mask.sum())}")
else:
    # Fallback: one fixed holdout shared by all Targets.
    row_ids = np.arange(len(df))
    stratify_labels = None
    low_name = "traffic_structural_susceptibility_class_LOW_STRUCTURAL_SUSCEPTIBILITY"
    mod_name = "traffic_structural_susceptibility_class_MODERATE_STRUCTURAL_SUSCEPTIBILITY"
    if low_name in df.columns and mod_name in df.columns:
        low_values = get_numeric_column(df, low_name).fillna(0).to_numpy()
        mod_values = get_numeric_column(df, mod_name).fillna(0).to_numpy()
        traffic_proxy = np.full(len(df), 2, dtype=int)
        traffic_proxy[mod_values > 0] = 1
        traffic_proxy[low_values > 0] = 0
        counts = pd.Series(traffic_proxy).value_counts()
        if len(counts) > 1 and counts.min() >= 2:
            stratify_labels = traffic_proxy
    tr_ids, te_ids = train_test_split(
        row_ids, test_size=TEST_SIZE_IF_NO_SPLIT, random_state=RANDOM_STATE,
        stratify=stratify_labels
    )
    train_mask = pd.Series(False, index=df.index)
    test_mask = pd.Series(False, index=df.index)
    train_mask.iloc[tr_ids] = True
    test_mask.iloc[te_ids] = True
    print(f"No usable split column; created a fixed holdout: train={int(train_mask.sum())}, test={int(test_mask.sum())}")

# ================================
# هندسة خصائص حتمية من أعمدة المصدر
# ================================
road_major_cols = ["roads_motorway_length_km", "roads_trunk_length_km", "roads_primary_road_length_km", "roads_secondary_road_length_km"]
road_local_cols = ["roads_tertiary_road_length_km", "roads_residential_road_length_km", "roads_service_road_length_km"]

def row_sum_available(frame, cols):
    values = pd.concat([get_numeric_column(frame, c) for c in cols], axis=1)
    return values.sum(axis=1, min_count=1)

df["fe_major_roads_sum"] = row_sum_available(df, road_major_cols)
df["fe_local_roads_sum"] = row_sum_available(df, road_local_cols)
df["fe_pop_x_road_density"] = get_numeric_column(df, "population_population_density_worldpop_2026") * get_numeric_column(df, "roads_road_density_km_per_km2")
df["fe_pop_x_intersections"] = get_numeric_column(df, "population_population_density_worldpop_2026") * get_numeric_column(df, "roads_intersection_density")
df["fe_ndvi_seasonal_diff"] = get_numeric_column(df, "green_space_ndvi_mean_annual_2026") - get_numeric_column(df, "green_space_ndvi_mean_summer_2026")

# ================================
# بناء Targets الخمسة
# ================================
targets_dict = {}

low_col = "traffic_structural_susceptibility_class_LOW_STRUCTURAL_SUSCEPTIBILITY"
mod_col = "traffic_structural_susceptibility_class_MODERATE_STRUCTURAL_SUSCEPTIBILITY"
if low_col not in df.columns or mod_col not in df.columns:
    raise KeyError(f"الأعمدة المطلوبة لبناء Target المرور غير موجودة: {low_col}, {mod_col}")
low_values = get_numeric_column(df, low_col).fillna(0).to_numpy()
mod_values = get_numeric_column(df, mod_col).fillna(0).to_numpy()
y_traffic = np.full(len(df), 2, dtype=int)  # 2 = High افتراضيًا، كما في الـ Notebook الأصلي
y_traffic[mod_values > 0] = 1                # 1 = Moderate
y_traffic[low_values > 0] = 0                # 0 = Low; له الأولوية إذا كان العمودان > 0
targets_dict["1_Traffic_Susceptibility_Class"] = pd.Series(y_traffic, index=df.index, dtype="Int64")

transit_presence = get_numeric_column(df, "transit_has_transit")
y = (transit_presence > 0).astype(float)
y[transit_presence.isna()] = np.nan
targets_dict["2_Transit_Availability_Binary"] = pd.Series(y, index=df.index, dtype="Float64")

acc_col = "transit_transit_accessibility_score"
acc_values = get_numeric_column(df, acc_col)
acc_median = acc_values.loc[train_mask].median()
if pd.isna(acc_median):
    raise ValueError(f"لا يمكن حساب Median للعمود {acc_col} من بيانات التدريب.")
y = (acc_values > acc_median).astype(float)
y[acc_values.isna()] = np.nan
targets_dict["3_High_Transit_Accessibility"] = pd.Series(y, index=df.index, dtype="Float64")

public_green = get_numeric_column(df, "green_space_has_public_green_space")
y = (public_green > 0).astype(float)
y[public_green.isna()] = np.nan
targets_dict["4_Public_Green_Space_Binary"] = pd.Series(y, index=df.index, dtype="Float64")

no2_col = "air_quality_s5p_no2_tropospheric_column_mean_umol_m2"
no2_values = get_numeric_column(df, no2_col)
no2_median = no2_values.loc[train_mask].median()
if pd.isna(no2_median):
    raise ValueError(f"لا يمكن حساب Median للعمود {no2_col} من بيانات التدريب.")
y = (no2_values > no2_median).astype(float)
y[no2_values.isna()] = np.nan
targets_dict["5_High_NO2_Air_Pollution"] = pd.Series(y, index=df.index, dtype="Float64")

# Feature lists: مأخوذة من النسخة المحسنة السابقة، مع السماح بإهمال أي عمود غير موجود وعرضه كتحذير.
features_per_target = {'1_Traffic_Susceptibility_Class': ['population_population_worldpop_2026', 'population_population_density_worldpop_2026', 'population_area_km2', 'population_cairo_intersection_ratio', 'population_latitude', 'population_longitude', 'roads_road_length_km', 'roads_road_density_km_per_km2', 'roads_motorway_length_km', 'roads_trunk_length_km', 'roads_primary_road_length_km', 'roads_secondary_road_length_km', 'roads_tertiary_road_length_km', 'roads_residential_road_length_km', 'roads_service_road_length_km', 'roads_other_road_length_km', 'roads_intersection_count', 'roads_intersection_density', 'roads_dead_end_count', 'roads_lit_road_ratio', 'roads_maxspeed_coverage', 'traffic_population_per_road_km', 'traffic_intersection_per_road_km', 'transit_has_transit', 'transit_stop_density_per_km2', 'transit_transit_accessibility_score', 'green_space_total_green_space_fraction', 'fe_major_roads_sum', 'fe_local_roads_sum', 'fe_pop_x_road_density', 'fe_pop_x_intersections'], '2_Transit_Availability_Binary': ['population_population_worldpop_2026', 'population_population_density_worldpop_2026', 'population_area_km2', 'population_cairo_intersection_ratio', 'population_latitude', 'population_longitude', 'roads_road_length_km', 'roads_road_density_km_per_km2', 'roads_motorway_length_km', 'roads_trunk_length_km', 'roads_primary_road_length_km', 'roads_secondary_road_length_km', 'roads_tertiary_road_length_km', 'roads_residential_road_length_km', 'roads_service_road_length_km', 'roads_intersection_count', 'roads_intersection_density', 'roads_dead_end_ratio', 'roads_oneway_ratio', 'roads_bridge_ratio', 'roads_lit_road_ratio', 'roads_maxspeed_coverage', 'traffic_population_per_road_km', 'traffic_structural_network_friction_index', 'green_space_ndvi_mean_annual_2026', 'green_space_total_green_space_fraction', 'green_space_agricultural_area_m2', 'fe_major_roads_sum', 'fe_local_roads_sum', 'fe_pop_x_road_density', 'fe_pop_x_intersections'], '3_High_Transit_Accessibility': ['transit_metro_station_count', 'transit_light_rail_station_count', 'transit_rail_station_count', 'transit_brt_station_count', 'transit_bus_stop_count', 'transit_paratransit_stop_count', 'transit_interchange_station_count', 'transit_multimodal_hub_present', 'transit_nearest_transit_stop_network_distance_m', 'transit_nearest_metro_station_network_distance_m', 'transit_nearest_brt_station_network_distance_m', 'population_population_worldpop_2026', 'population_population_density_worldpop_2026', 'population_cairo_intersection_ratio', 'population_latitude', 'population_longitude', 'roads_road_density_km_per_km2', 'roads_primary_road_length_km', 'roads_secondary_road_length_km', 'roads_residential_road_length_km', 'roads_intersection_density', 'roads_dead_end_ratio', 'roads_lit_road_ratio', 'traffic_structural_network_friction_index', 'fe_major_roads_sum', 'fe_pop_x_intersections'], '4_Public_Green_Space_Binary': ['green_space_ndvi_mean_summer_2026', 'green_space_ndvi_median_summer_2026', 'green_space_ndvi_p90_summer_2026', 'green_space_ndvi_min_summer_2026', 'green_space_vegetated_fraction_summer_2026', 'green_space_ndvi_mean_annual_2026', 'green_space_ndvi_median_annual_2026', 'green_space_ndvi_p90_annual_2026', 'green_space_ndvi_min_annual_2026', 'green_space_vegetated_fraction_annual_2026', 'green_space_vegetation_persistence_ratio', 'green_space_private_restricted_green_space_area_m2', 'green_space_agricultural_area_m2', 'green_space_natural_reserve_area_m2', 'population_population_worldpop_2026', 'population_population_density_worldpop_2026', 'population_cairo_intersection_ratio', 'population_latitude', 'population_longitude', 'roads_road_density_km_per_km2', 'roads_primary_road_length_km', 'roads_residential_road_length_km', 'roads_intersection_density', 'roads_lit_road_ratio', 'transit_transit_accessibility_score', 'fe_ndvi_seasonal_diff', 'fe_pop_x_road_density'], '5_High_NO2_Air_Pollution': ['roads_motorway_length_km', 'roads_trunk_length_km', 'roads_primary_road_length_km', 'roads_secondary_road_length_km', 'roads_tertiary_road_length_km', 'roads_residential_road_length_km', 'roads_road_density_km_per_km2', 'roads_intersection_density', 'roads_oneway_ratio', 'roads_bridge_ratio', 'traffic_structural_network_friction_index', 'traffic_population_per_road_km', 'population_population_worldpop_2026', 'population_population_density_worldpop_2026', 'population_cairo_intersection_ratio', 'population_latitude', 'population_longitude', 'transit_bus_stop_count', 'transit_paratransit_stop_count', 'transit_stop_density_per_km2', 'transit_transit_accessibility_score', 'green_space_ndvi_mean_annual_2026', 'green_space_vegetated_fraction_annual_2026', 'green_space_total_green_space_fraction', 'green_space_agricultural_area_m2', 'air_quality_nearest_pm25_station_distance_km', 'air_quality_pm25_station_count_within_5km', 'fe_major_roads_sum', 'fe_pop_x_road_density', 'fe_pop_x_intersections']}

print("\nTargets ready:")
for target_name, y_series in targets_dict.items():
    valid = y_series.dropna().astype(int)
    print(f"- {target_name}: n={len(valid):,}; distribution={valid.value_counts().sort_index().to_dict()}")


Using existing split column: train=2422, test=1067

Targets ready:
- 1_Traffic_Susceptibility_Class: n=3,489; distribution={0: 1264, 1: 1074, 2: 1151}
- 2_Transit_Availability_Binary: n=3,489; distribution={0: 3051, 1: 438}
- 3_High_Transit_Accessibility: n=3,489; distribution={0: 2798, 1: 691}
- 4_Public_Green_Space_Binary: n=3,489; distribution={0: 3101, 1: 388}
- 5_High_NO2_Air_Pollution: n=3,489; distribution={0: 1745, 1: 1744}


## 3. قائمة الموديلات والـ Parameters

التجربة بتلف على كل Target، وبعدها على كل Model في القائمة، وبعدها على كل إعداد داخل `parameter_sets`. إضافة موديل جديد تكون بإضافة عنصر ثالث في `MODEL_CONFIGS`، مع Class الموديل وقائمة الـ Parameters الخاصة به.

**ملاحظة بخصوص SMOTE:** يتم تطبيقه على بيانات الـ fit فقط، داخل الـ Pipeline. الإعدادات الافتراضية هنا تستخدم `k_neighbors` أصغر تلقائيًا إذا كانت أقل فئة صغيرة. لأن Features المشروع رقمية وفيها أعمدة مؤشرات ثنائية أيضًا، راجع طبيعة القيم الاصطناعية الناتجة؛ SMOTE القياسي قد ينتج قيمًا وسطية للأعمدة الثنائية.


In [ ]:
# غيّر القوائم أو أضف إعدادات جديدة هنا — كل dict يمثل تجربة مستقلة.
MODEL_CONFIGS = [
    {
        "name": "Logistic Regression",
        "estimator": LogisticRegression,
        "parameter_sets": [
            # أقرب إعداد للنسخة المحسنة السابقة
            {"C": 0.5, "max_iter": 1500, "solver": "lbfgs", "class_weight": "balanced", "random_state": RANDOM_STATE},
            {"C": 0.1, "max_iter": 3000, "solver": "lbfgs", "class_weight": None, "random_state": RANDOM_STATE},
            {"C": 0.5, "max_iter": 3000, "solver": "lbfgs", "class_weight": None, "random_state": RANDOM_STATE},
            {"C": 1.0, "max_iter": 3000, "solver": "lbfgs", "class_weight": None, "random_state": RANDOM_STATE},
        ],
    },
    {
        "name": "MLP Classifier",
        "estimator": MLPClassifier,
        "parameter_sets": [
            # أقرب إعداد للنسخة المحسنة السابقة
            {"hidden_layer_sizes": (128, 64), "activation": "relu", "solver": "adam", "alpha": 0.05, "learning_rate": "adaptive", "early_stopping": True, "validation_fraction": 0.15, "max_iter": 400, "random_state": RANDOM_STATE},
            {"hidden_layer_sizes": (64, 32), "activation": "relu", "solver": "adam", "alpha": 0.01, "learning_rate_init": 0.001, "early_stopping": True, "max_iter": 500, "random_state": RANDOM_STATE},
            {"hidden_layer_sizes": (128,), "activation": "relu", "solver": "adam", "alpha": 0.001, "learning_rate_init": 0.0005, "early_stopping": True, "max_iter": 600, "random_state": RANDOM_STATE},
        ],
    },
]
# مثال لإضافة موديل ثالث: شيل التعليقات وأضفه داخل MODEL_CONFIGS.
# MODEL_CONFIGS.append({
#     "name": "Random Forest",
#     "estimator": RandomForestClassifier,
#     "parameter_sets": [
#         {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1, "random_state": RANDOM_STATE, "n_jobs": -1},
#         {"n_estimators": 400, "max_depth": 15, "min_samples_leaf": 2, "random_state": RANDOM_STATE, "n_jobs": -1},
#     ],
# })

# تحقق مبدئي من أن كل موديل عنده إعداد واحد على الأقل
assert MODEL_CONFIGS, "MODEL_CONFIGS لا تحتوي على موديلات."
for spec in MODEL_CONFIGS:
    assert spec.get("parameter_sets"), f"لا توجد parameter_sets للموديل {spec.get('name')}"

print("Models / candidate parameter sets:")
for spec in MODEL_CONFIGS:
    print(f"- {spec['name']}: {len(spec['parameter_sets'])} configurations")


Models / candidate parameter sets:
- Logistic Regression: 4 configurations
- MLP Classifier: 3 configurations


## 4. التدريب والاختيار والتقييم النهائي

- **Fit subset:** لتدريب الإعدادات المرشحة.
- **Validation subset:** اختيار الإعداد الأعلى في Macro-F1.
- **Full Training:** إعادة تدريب أفضل إعداد على كامل مجموعة التدريب الأصلية؛ ويعمل SMOTE داخل الـ Pipeline على هذه المجموعة فقط.
- **Test:** تقييم نهائي مرة واحدة دون استخدامه لاختيار Parameters أو Threshold.

لن يتم ضبط Threshold باستخدام الـ Test. المقاييس المطبوعة تشمل Accuracy وMacro-F1 وWeighted-F1 وPrecision وRecall وROC-AUC وتقارير الفئات وConfusion Matrix.


In [ ]:
def make_pipeline(estimator_class, params, y_fit, n_input_features, use_smote=True):
    """Build leakage-safe imputation, optional polynomial/selection, scaling, SMOTE, and estimator pipeline."""
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if USE_POLYNOMIAL_FEATURES:
        # Number of degree-2 terms without bias: n original + n*(n+1)/2 pair/square terms.
        poly_feature_count = n_input_features + (n_input_features * (n_input_features + 1) // 2)
        selected_k = min(SELECT_K_BEST, poly_feature_count)
        steps.append(("poly", PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)))
        steps.append(("selector", SelectKBest(score_func=f_classif, k=selected_k)))
    steps.append(("scaler", StandardScaler()))
    if use_smote:
        class_counts = pd.Series(np.asarray(y_fit, dtype=int)).value_counts()
        if len(class_counts) < 2:
            raise ValueError("SMOTE يحتاج فئتين على الأقل في بيانات التدريب.")
        min_count = int(class_counts.min())
        if min_count < 2:
            raise ValueError(f"أصغر فئة تحتوي على {min_count} عينة فقط؛ لا يمكن تطبيق SMOTE بأمان.")
        k_neighbors = min(SMOTE_K_NEIGHBORS, min_count - 1)
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)))
    steps.append(("model", estimator_class(**params)))
    return ImbPipeline(steps=steps)


def calculate_roc_auc(model, X_test, y_test):
    """Calculate binary/multiclass ROC-AUC when the estimator exposes probabilities."""
    if not hasattr(model, "predict_proba"):
        return np.nan
    try:
        probabilities = model.predict_proba(X_test)
        classes = np.asarray(model.classes_)
        unique_y = np.sort(pd.Series(y_test).unique())
        if len(unique_y) == 2:
            positive_label = 1 if 1 in classes else classes[-1]
            positive_index = int(np.where(classes == positive_label)[0][0])
            return roc_auc_score(y_test, probabilities[:, positive_index])
        return roc_auc_score(y_test, probabilities, labels=classes, multi_class="ovr", average="macro")
    except Exception:
        return np.nan


def serialize_params(params):
    return json.dumps(params, default=lambda obj: list(obj) if isinstance(obj, tuple) else str(obj), ensure_ascii=False)

all_results = []
best_params_records = []
all_test_predictions = pd.DataFrame(index=df.index)
run_reports = {}

for target_name, y_series in targets_dict.items():
    print("\n" + "=" * 100)
    print(f"TARGET: {target_name}")
    print("=" * 100)

    # Skip rows whose label for this Target is missing.
    valid_mask = y_series.notna()
    target_index = df.index[valid_mask]
    y_all = y_series.loc[target_index].astype(int)

    # Use only real features listed for this target and present in the dataset.
    requested_cols = features_per_target[target_name]
    available_cols = [col for col in requested_cols if col in df.columns]
    missing_cols = [col for col in requested_cols if col not in df.columns]
    if missing_cols:
        print(f"WARNING: {len(missing_cols)} configured feature(s) missing and skipped: {missing_cols}")
    if not available_cols:
        print(f"SKIP {target_name}: none of its configured features were found.")
        continue

    X_all = df.loc[target_index, available_cols].copy()
    for col in X_all.columns:
        X_all[col] = pd.to_numeric(X_all[col], errors="coerce")

    this_train_mask = train_mask.loc[target_index].astype(bool)
    this_test_mask = test_mask.loc[target_index].astype(bool)
    X_train_full = X_all.loc[this_train_mask]
    y_train_full = y_all.loc[this_train_mask]
    X_test = X_all.loc[this_test_mask]
    y_test = y_all.loc[this_test_mask]

    if len(X_train_full) == 0 or len(X_test) == 0:
        print(f"SKIP {target_name}: empty train or test partition.")
        continue
    if y_train_full.nunique() < 2 or y_test.nunique() < 2:
        print(f"SKIP {target_name}: train/test must each contain at least 2 classes.")
        continue

    min_train_class = int(y_train_full.value_counts().min())
    val_size = VALIDATION_SIZE
    # Need enough examples of each class on both sides for stratified validation + SMOTE.
    stratify_arg = y_train_full if min_train_class >= 2 else None
    try:
        X_fit, X_val, y_fit, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size, random_state=RANDOM_STATE,
            stratify=stratify_arg
        )
    except ValueError:
        X_fit, X_val, y_fit, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size, random_state=RANDOM_STATE,
            stratify=None
        )

    print(f"Features: {len(available_cols)} | Fit: {len(X_fit)} | Validation: {len(X_val)} | Final Train: {len(X_train_full)} | Test: {len(X_test)}")
    print(f"Train class counts before SMOTE: {y_train_full.value_counts().sort_index().to_dict()}")
    print(f"Test class counts: {y_test.value_counts().sort_index().to_dict()}")

    for spec in MODEL_CONFIGS:
        model_name = spec["name"]
        estimator_class = spec["estimator"]
        best_val_f1 = -np.inf
        best_params = None
        failed_configs = []
        tested_configs = []

        print(f"\n--- Tuning: {model_name} ---")
        for config_id, params in enumerate(spec["parameter_sets"], start=1):
            try:
                candidate = make_pipeline(estimator_class, params, y_fit, X_fit.shape[1], use_smote=True)
                candidate.fit(X_fit, y_fit)
                val_pred = candidate.predict(X_val)
                val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
                val_accuracy = accuracy_score(y_val, val_pred)
                tested_configs.append({
                    "config_id": config_id,
                    "params": params,
                    "validation_macro_f1": float(val_macro_f1),
                    "validation_accuracy": float(val_accuracy),
                })
                print(f"  config {config_id}/{len(spec['parameter_sets'])}: Validation Macro-F1={val_macro_f1:.4f} | Accuracy={val_accuracy:.4f} | params={params}")
                if val_macro_f1 > best_val_f1:
                    best_val_f1 = val_macro_f1
                    best_params = params
            except Exception as exc:
                failed_configs.append({"config_id": config_id, "params": params, "error": str(exc)})
                print(f"  config {config_id} FAILED: {type(exc).__name__}: {exc}")

        if best_params is None:
            print(f"SKIP model {model_name} for {target_name}: all configurations failed.")
            run_reports[f"{target_name} | {model_name}"] = {"failed_configs": failed_configs}
            continue

        # Refit selected configuration on every row in the original Training partition.
        final_model = make_pipeline(estimator_class, best_params, y_train_full, X_train_full.shape[1], use_smote=True)
        final_model.fit(X_train_full, y_train_full)
        test_pred = final_model.predict(X_test)
        test_acc = accuracy_score(y_test, test_pred)
        test_macro_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)
        test_weighted_f1 = f1_score(y_test, test_pred, average="weighted", zero_division=0)
        test_macro_precision = precision_score(y_test, test_pred, average="macro", zero_division=0)
        test_macro_recall = recall_score(y_test, test_pred, average="macro", zero_division=0)
        test_auc = calculate_roc_auc(final_model, X_test, y_test)

        print(f"\nBEST: {model_name} | {target_name}")
        print(f"Best validation Macro-F1: {best_val_f1:.4f}")
        print(f"Test Accuracy={test_acc:.4f} | Macro-F1={test_macro_f1:.4f} | Weighted-F1={test_weighted_f1:.4f} | Macro-Precision={test_macro_precision:.4f} | Macro-Recall={test_macro_recall:.4f} | ROC-AUC={test_auc:.4f}" if pd.notna(test_auc) else f"Test Accuracy={test_acc:.4f} | Macro-F1={test_macro_f1:.4f} | Weighted-F1={test_weighted_f1:.4f} | Macro-Precision={test_macro_precision:.4f} | Macro-Recall={test_macro_recall:.4f} | ROC-AUC=N/A")
        print("Classification report (held-out Test):")
        report_text = classification_report(y_test, test_pred, digits=4, zero_division=0)
        print(report_text)
        cm = confusion_matrix(y_test, test_pred, labels=final_model.classes_)
        print(f"Confusion matrix (labels={list(final_model.classes_)}):\n{cm}")

        all_results.append({
            "Target": target_name,
            "Model": model_name,
            "Features_Used": len(available_cols),
            "Fit_Rows_For_Selection": len(X_fit),
            "Validation_Rows": len(X_val),
            "Full_Training_Rows": len(X_train_full),
            "Test_Rows": len(X_test),
            "Best_Validation_Macro_F1": round(float(best_val_f1), 4),
            "Test_Accuracy": round(float(test_acc), 4),
            "Test_Macro_F1": round(float(test_macro_f1), 4),
            "Test_Weighted_F1": round(float(test_weighted_f1), 4),
            "Test_Macro_Precision": round(float(test_macro_precision), 4),
            "Test_Macro_Recall": round(float(test_macro_recall), 4),
            "Test_ROC_AUC": round(float(test_auc), 4) if pd.notna(test_auc) else np.nan,
            "Best_Params": serialize_params(best_params),
        })
        best_params_records.append({
            "Target": target_name,
            "Model": model_name,
            "Best_Validation_Macro_F1": float(best_val_f1),
            "Best_Params": best_params,
            "Candidate_Results": tested_configs,
            "Failed_Configurations": failed_configs,
        })
        run_reports[f"{target_name} | {model_name}"] = {
            "best_params": best_params,
            "best_validation_macro_f1": float(best_val_f1),
            "test_classification_report": classification_report(y_test, test_pred, output_dict=True, zero_division=0),
            "test_confusion_matrix": cm.tolist(),
            "test_labels_order": [int(x) for x in final_model.classes_],
            "candidate_results": tested_configs,
            "failed_configurations": failed_configs,
        }

        # Keep target and predictions aligned with original row indices for auditing/export.
        all_test_predictions.loc[X_test.index, f"{target_name}__true"] = y_test.astype(int)
        all_test_predictions.loc[X_test.index, f"{target_name}__{model_name.replace(' ', '_')}__pred"] = test_pred.astype(int)
        if hasattr(final_model, "predict_proba"):
            try:
                probabilities = final_model.predict_proba(X_test)
                for class_idx, class_label in enumerate(final_model.classes_):
                    all_test_predictions.loc[X_test.index, f"{target_name}__{model_name.replace(' ', '_')}__proba_{class_label}"] = probabilities[:, class_idx]
            except Exception:
                pass

# ================================
# الملخص والتصدير
# ================================
if not all_results:
    raise RuntimeError("لم تُنتج أي نتائج. راجع أسماء الأعمدة وقوائم Features والـ Targets.")

results_df = pd.DataFrame(all_results).sort_values(["Target", "Model"]).reset_index(drop=True)
print("\n" + "#" * 100)
print("FINAL SMOTE MODEL COMPARISON — METRICS FROM HELD-OUT TEST")
print("#" * 100)
display(results_df)

# اختيار النموذج المعروض لكل Target يكون بناءً على Validation فقط؛ لا نختار نموذج النشر اعتمادًا على Test.
best_per_target_validation = results_df.loc[results_df.groupby("Target")["Best_Validation_Macro_F1"].idxmax()].sort_values("Target")
print("\nModel per Target with the best Validation Macro-F1 (Test metrics remain an independent final report):")
display(best_per_target_validation[["Target", "Model", "Best_Validation_Macro_F1", "Test_Accuracy", "Test_Macro_F1", "Test_ROC_AUC", "Best_Params"]])

results_df.to_csv("smote_model_comparison.csv", index=False)
all_test_predictions.to_csv("smote_test_predictions.csv", index_label="row_index")
with open("smote_best_params_and_reports.json", "w", encoding="utf-8") as f:
    json.dump(best_params_records, f, ensure_ascii=False, indent=2, default=lambda obj: list(obj) if isinstance(obj, tuple) else (obj.item() if isinstance(obj, np.generic) else str(obj)))
print("\nSaved files:")
print("- smote_model_comparison.csv")
print("- smote_test_predictions.csv")
print("- smote_best_params_and_reports.json")



TARGET: 1_Traffic_Susceptibility_Class
Features: 31 | Fit: 1937 | Validation: 485 | Final Train: 2422 | Test: 1067
Train class counts before SMOTE: {0: 888, 1: 727, 2: 807}
Test class counts: {0: 376, 1: 347, 2: 344}

--- Tuning: Logistic Regression ---
  config 1/4: Validation Macro-F1=0.8878 | Accuracy=0.8928 | params={'C': 0.5, 'max_iter': 1500, 'solver': 'lbfgs', 'class_weight': 'balanced', 'random_state': 42}
  config 2/4: Validation Macro-F1=0.8747 | Accuracy=0.8784 | params={'C': 0.1, 'max_iter': 3000, 'solver': 'lbfgs', 'class_weight': None, 'random_state': 42}
  config 3/4: Validation Macro-F1=0.8878 | Accuracy=0.8928 | params={'C': 0.5, 'max_iter': 3000, 'solver': 'lbfgs', 'class_weight': None, 'random_state': 42}
  config 4/4: Validation Macro-F1=0.8920 | Accuracy=0.8969 | params={'C': 1.0, 'max_iter': 3000, 'solver': 'lbfgs', 'class_weight': None, 'random_state': 42}

BEST: Logistic Regression | 1_Traffic_Susceptibility_Class
Best validation Macro-F1: 0.8920
Test Accuracy=

,Target,Model,Features_Used,Fit_Rows_For_Selection,Validation_Rows,Full_Training_Rows,Test_Rows,Best_Validation_Macro_F1,Test_Accuracy,Test_Macro_F1,Test_Weighted_F1,Test_Macro_Precision,Test_Macro_Recall,Test_ROC_AUC,Best_Params
0,1_Traffic_Susceptibility_Class,Logistic Regression,31,1937,485,2422,1067,0.8920,0.9110,0.9090,0.9111,0.9112,0.9088,0.9826,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
1,1_Traffic_Susceptibility_Class,MLP Classifier,31,1937,485,2422,1067,0.8811,0.9082,0.9072,0.9091,0.9102,0.9067,0.9815,"{""hidden_layer_sizes"": [64, 32], ""activation"":..."
2,2_Transit_Availability_Binary,Logistic Regression,31,1937,485,2422,1067,0.8353,0.9007,0.8119,0.9113,0.7649,0.9120,0.9737,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
3,2_Transit_Availability_Binary,MLP Classifier,31,1937,485,2422,1067,0.8374,0.9138,0.8264,0.9212,0.7835,0.9018,0.9736,"{""hidden_layer_sizes"": [128], ""activation"": ""r..."
4,3_High_Transit_Accessibility,Logistic Regression,26,1937,485,2422,1067,0.9685,0.9822,0.9712,0.9823,0.9668,0.9757,0.9981,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
5,3_High_Transit_Accessibility,MLP Classifier,26,1937,485,2422,1067,0.9652,0.9775,0.9639,0.9777,0.9556,0.9728,0.9968,"{""hidden_layer_sizes"": [64, 32], ""activation"":..."
6,4_Public_Green_Space_Binary,Logistic Regression,27,1937,485,2422,1067,0.7408,0.8613,0.7494,0.8814,0.7076,0.8834,0.9301,"{""C"": 0.1, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
7,4_Public_Green_Space_Binary,MLP Classifier,27,1937,485,2422,1067,0.7223,0.8632,0.7370,0.8806,0.6987,0.8377,0.9236,"{""hidden_layer_sizes"": [128], ""activation"": ""r..."
8,5_High_NO2_Air_Pollution,Logistic Regression,30,1937,485,2422,1067,0.8660,0.8613,0.8610,0.8611,0.8638,0.8613,0.9298,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
9,5_High_NO2_Air_Pollution,MLP Classifier,30,1937,485,2422,1067,0.8804,0.8688,0.8686,0.8686,0.8707,0.8688,0.9384,"{""hidden_layer_sizes"": [64, 32], ""activation"":..."



Model per Target with the best Validation Macro-F1 (Test metrics remain an independent final report):


,Target,Model,Best_Validation_Macro_F1,Test_Accuracy,Test_Macro_F1,Test_ROC_AUC,Best_Params
0,1_Traffic_Susceptibility_Class,Logistic Regression,0.8920,0.9110,0.9090,0.9826,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
3,2_Transit_Availability_Binary,MLP Classifier,0.8374,0.9138,0.8264,0.9736,"{""hidden_layer_sizes"": [128], ""activation"": ""r..."
4,3_High_Transit_Accessibility,Logistic Regression,0.9685,0.9822,0.9712,0.9981,"{""C"": 1.0, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
6,4_Public_Green_Space_Binary,Logistic Regression,0.7408,0.8613,0.7494,0.9301,"{""C"": 0.1, ""max_iter"": 3000, ""solver"": ""lbfgs""..."
9,5_High_NO2_Air_Pollution,MLP Classifier,0.8804,0.8688,0.8686,0.9384,"{""hidden_layer_sizes"": [64, 32], ""activation"":..."



Saved files:
- smote_model_comparison.csv
- smote_test_predictions.csv
- smote_best_params_and_reports.json


## 5. مقارنة نتائج SMOTE الجديدة بالمرجع القديم

النتائج المرجعية ليست baseline عادلة 100% للمقارنة، لأن النسخة القديمة استخدمت preprocessing وهندسة خصائص مختلفين. استخدم القسم التالي لعرض المرجع بجانب نتائج التشغيل الحالي بعد اكتمال التدريب.


In [ ]:
# Compare historical reference to new run (new columns stay explicitly labeled as test metrics).
reference_df = pd.DataFrame(previous_best_results).rename(columns={
    "Previous_Best_Model": "Previous_Model",
    "Accuracy": "Previous_Accuracy",
    "Macro_F1": "Previous_Macro_F1",
    "ROC_AUC": "Previous_ROC_AUC",
})
new_best = results_df.loc[results_df.groupby("Target")["Best_Validation_Macro_F1"].idxmax(), [
    "Target", "Model", "Best_Validation_Macro_F1", "Test_Accuracy", "Test_Macro_F1", "Test_ROC_AUC"
]].rename(columns={
    "Model": "New_SMOTE_Model_Selected_By_Validation",
    "Best_Validation_Macro_F1": "New_Best_Validation_Macro_F1",
    "Test_Accuracy": "New_Test_Accuracy",
    "Test_Macro_F1": "New_Test_Macro_F1",
    "Test_ROC_AUC": "New_Test_ROC_AUC",
})
comparison_df = reference_df.merge(new_best, on="Target", how="left")
display(comparison_df)
comparison_df.to_csv("smote_vs_previous_reference.csv", index=False)
print("Saved: smote_vs_previous_reference.csv")


,Target,Previous_Model,Previous_Accuracy,Previous_Macro_F1,Previous_ROC_AUC,New_SMOTE_Model_Selected_By_Validation,New_Best_Validation_Macro_F1,New_Test_Accuracy,New_Test_Macro_F1,New_Test_ROC_AUC
0,1_Traffic_Susceptibility_Class,Logistic Regression (Tuned + Poly),0.9126,0.9135,0.9837,Logistic Regression,0.8920,0.9110,0.9090,0.9826
1,2_Transit_Availability_Binary,Logistic Regression (Tuned + Poly),0.9393,0.8908,0.9753,MLP Classifier,0.8374,0.9138,0.8264,0.9736
2,3_High_Transit_Accessibility,Logistic Regression (Tuned + Poly),0.9842,0.9781,0.9981,Logistic Regression,0.9685,0.9822,0.9712,0.9981
3,4_Public_Green_Space_Binary,Logistic Regression (Tuned + Poly),0.8935,0.7590,0.9310,Logistic Regression,0.7408,0.8613,0.7494,0.9301
4,5_High_NO2_Air_Pollution,MLP Classifier (baseline features),0.9418,0.9417,0.9779,MLP Classifier,0.8804,0.8688,0.8686,0.9384


Saved: smote_vs_previous_reference.csv
